MODELOS


In [ ]:
#DINOV3
from mapminer.models import DINOv3
model = DINOv3(pretrained=True)
x = normalize(input_tensor)
output = model(x)

In [ ]:
#NAFNET
from mapminer.models import NAFNet
model = NAFNet(in_channels=12, dim=32)
output = model(input_tensor)

In [ ]:
#SAM3
from mapminer.models import SAM3
sam3 = SAM3() 
df = sam3.inference(ds,text='building', exemplars=None)

MINEROS

In [ ]:
#GOOGLE BASEMAPMINER
from mapminer.miners import GoogleBaseMapMiner
miner = GoogleBaseMapMiner()
ds = miner.fetch(lat=40.748817, lon=-73.985428, radius=500)

In [ ]:
#CDLMiner
from mapminer.miners import CDLMiner
miner = CDLMiner()
ds = miner.fetch(lon=-95.665, lat=39.8283, radius=10000, daterange="2024-01-01/2024-01-10")

In [ ]:
#GOOGLE BUILDING MINER
from mapminer.miners import GoogleBuildingMiner
miner = GoogleBuildingMiner()
ds = miner.fetch(lat=34.052235, lon=-118.243683, radius=1000)

VISUALIZAR DATA

In [ ]:
import hvplot.xarray
ds.hvplot.image(title=f"Captured on {ds.attrs['metadata']['date']['value']}")

EMPEZAMOS


In [1]:
from mapminer.miners import *
import holoviews
import panel as pn
import hvplot.xarray
import hvplot.pandas
import holoviews as hv

In [3]:
from mapminer.miners import LandsatMiner
import hvplot.xarray
import pandas as pd

# 1. Definir el área y obtener los datos
miner = LandsatMiner()
# NOTA: Ajusta estas coordenadas a tu campo de conreo específico
# Ampliamos el radio un poco para capturar bien el terreno
ds = miner.fetch(lat=39.8283, lon=-95.665, radius=1000, daterange="2023-07-01/2023-07-15")

# 2. Filtrar el Dataset para quedarnos SOLO con las bandas que nos interesan
# Landsat 8/9 nombra estas bandas de la siguiente manera:
bandas_interes = ['red', 'green', 'blue', 'nir08', 'swir16', 'swir22']
ds_multispectral = ds[bandas_interes]

# 4. VISUALIZAR LA IMAGEN RGB (Color Verdadero)
# Juntamos las tres variables en un solo DataArray con una dimensión llamada 'band'
rgb_array = ds_single_time[['red', 'green', 'blue']].to_array(dim='band')

# Ahora le decimos a hvplot que use esa dimensión 'band' para los canales RGB
plot_rgb = rgb_array.hvplot.rgb(
    x='x', 
    y='y', 
    bands='band', 
    title=f"Campo de Conreo RGB - {ds_single_time.time.values.astype('datetime64[D]')}",
    width=600, 
    height=500,
    geo=True # Mantiene la proyección espacial
)
display(plot_rgb)

# 5. OBTENER LOS VALORES EN UN DATAFRAME
# Al llamar a to_dataframe() sobre el Dataset filtrado, Pandas crea las columnas automáticamente
# sin dar el error del "unnamed DataArray"
df_valores = ds_single_time.to_dataframe().reset_index()

# Limpiar los datos: eliminamos las filas donde las bandas tengan valores nulos (nodata en los bordes)
df_valores = df_valores.dropna(subset=bandas_interes)

# Mostrar los primeros resultados
print("Dimensiones del DataFrame:", df_valores.shape)
display(df_valores.head())

:RGB   [x,y]   (R,G,B)

Dimensiones del DataFrame: (3850, 10)


,y,x,red,green,blue,nir08,swir16,swir22,spatial_ref,time
0,4413135.0,271125.0,31473.0,30749.0,31926.0,35283.0,29299.0,27886.0,32615,2023-07-01 17:06:00.762128
1,4413135.0,271155.0,32164.0,31505.0,32272.0,35666.0,30075.0,28720.0,32615,2023-07-01 17:06:00.762128
2,4413135.0,271185.0,32007.0,31390.0,32213.0,35524.0,30199.0,28842.0,32615,2023-07-01 17:06:00.762128
3,4413135.0,271215.0,31794.0,31234.0,31941.0,35050.0,30029.0,28719.0,32615,2023-07-01 17:06:00.762128
4,4413135.0,271245.0,31981.0,31394.0,32056.0,35276.0,30221.0,29119.0,32615,2023-07-01 17:06:00.762128


In [5]:
# ==========================================
# 1. IMPORTACIONES NECESARIAS
# ==========================================
from mapminer.miners import LandsatMiner
import hvplot.xarray
import hvplot
import pandas as pd
import rioxarray

# ==========================================
# 2. DESCARGA Y PREPARACIÓN DE DATOS
# ==========================================
print("Descargando datos del satélite...")
miner = LandsatMiner()
# Coordenadas de tu campo de conreo
ds = miner.fetch(lat=39.8283, lon=-95.665, radius=1000, daterange="2023-07-01/2023-07-15")

# Filtramos solo las bandas multiespectrales que nos interesan
bandas_interes = ['red', 'green', 'blue', 'nir08', 'swir16', 'swir22']
ds_multispectral = ds[bandas_interes]

# Seleccionamos la primera captura temporal disponible
ds_single_time = ds_multispectral.isel(time=0)
fecha_captura = ds_single_time.time.values.astype('datetime64[D]')

# ==========================================
# 3. VISUALIZACIÓN RGB Y GUARDADO DE IMAGEN (PNG)
# ==========================================
print(f"Generando imagen RGB para la fecha {fecha_captura}...")
# Agrupamos las bandas RGB en una dimensión para hvplot
rgb_array = ds_single_time[['red', 'green', 'blue']].to_array(dim='band')

# Generamos el gráfico con clim para ajustar el brillo (los valores de Landsat son altos)
plot_rgb = rgb_array.hvplot.rgb(
    x='x', 
    y='y', 
    bands='band', 
    clim=(7000, 35000), # Ajusta estos valores si la imagen sigue muy oscura o clara
    title=f"Campo de Conreo RGB - {fecha_captura}",
    width=600, 
    height=500,
    geo=True
)

# Mostramos el gráfico en el notebook
display(plot_rgb)

# Guardamos el gráfico como PNG
hvplot.save(plot_rgb, 'campo_conreo_visualizacion.html')
print("Imagen PNG guardada con éxito.")

# ==========================================
# 4. EXPORTACIÓN DE DATOS "A TOPE"
# ==========================================

# --- A. Guardar Tabla de Valores (CSV y Parquet) ---
print("Generando tabla de datos masivos...")
df_valores = ds_single_time.to_dataframe().reset_index()
df_valores = df_valores.dropna(subset=bandas_interes) # Limpiamos valores nulos

# Mostramos un resumen
print(f"Dimensiones del DataFrame: {df_valores.shape}")

# Guardamos en CSV (ideal para Excel/Python clásico)
df_valores.to_csv('datos_conreo_completos.csv', index=False)
# Guardamos en Parquet (ideal para cargar datos masivos súper rápido)
df_valores.to_parquet('datos_conreo_masivos.parquet')
print("Archivos CSV y Parquet guardados con éxito.")

# --- B. Guardar Imagen Original Satelital (GeoTIFF) ---
print("Guardando archivo GeoTIFF original...")
# Asignamos el sistema de coordenadas
crs_value = ds_single_time.spatial_ref.item()
ds_single_time.rio.write_crs(crs_value, inplace=True)

# Guardamos el raster con todas las bandas intactas para usar en QGIS/ArcGIS
ds_single_time.rio.to_raster('imagen_original_multiespectral.tif')
print("Archivo GeoTIFF guardado con éxito. ¡Proceso terminado!")

Descargando datos del satélite...
Generando imagen RGB para la fecha 2023-07-01...


:RGB   [x,y]   (R,G,B)

Imagen PNG guardada con éxito.
Generando tabla de datos masivos...
Dimensiones del DataFrame: (3850, 10)
Archivos CSV y Parquet guardados con éxito.
Guardando archivo GeoTIFF original...
Archivo GeoTIFF guardado con éxito. ¡Proceso terminado!


In [6]:
import pandas as pd

# Imagina que le añadiste una etiqueta a tus datos para saber qué campo es
df_valores['id_campo'] = 'Campo_Prueba_01'

# Definimos qué bandas queremos resumir
bandas = ['red', 'green', 'blue', 'nir08', 'swir16', 'swir22']

# Agrupamos por campo (y fecha, si tuvieras varias) y calculamos múltiples métricas a la vez
df_resumen = df_valores.groupby(['id_campo', 'time'])[bandas].agg(['mean', 'std', 'min', 'max'])

# Aplanamos el nombre de las columnas (para que no queden con niveles dobles)
df_resumen.columns = ['_'.join(col).strip() for col in df_resumen.columns.values]
df_resumen = df_resumen.reset_index()

display(df_resumen)

,id_campo,time,red_mean,red_std,red_min,red_max,green_mean,green_std,green_min,green_max,...,nir08_min,nir08_max,swir16_mean,swir16_std,swir16_min,swir16_max,swir22_mean,swir22_std,swir22_min,swir22_max
0,Campo_Prueba_01,2023-07-01 17:06:00.762128,17129.666016,6055.345215,4524.0,32164.0,17005.457031,5934.652832,5531.0,31505.0,...,19302.0,35668.0,21423.599609,3773.442139,12451.0,30459.0,19585.310547,4262.75,9500.0,29576.0


In [ ]:
import os
from mapminer.miners import LandsatMiner
import pandas as pd
import hvplot.xarray
import hvplot
import rioxarray

# ==========================================
# 1. CREACIÓN DE CARPETAS
# ==========================================
base_dir = "resultados_satelite"
carpetas = {
    "html": os.path.join(base_dir, "visualizaciones_html"),
    "tif": os.path.join(base_dir, "imagenes_originales_tif"),
    "csv": os.path.join(base_dir, "dataset_final")
}

for ruta in carpetas.values():
    os.makedirs(ruta, exist_ok=True)

print(f"Carpetas preparadas en: ./{base_dir}/")

# ==========================================
# 2. BASE DE DATOS 
# ==========================================
anclas_estrategicas = [
    # --- SANOS ---
    {"zona": "Cinturon_Maiz_Iowa", "lat": 41.55, "lon": -93.60, "estado": "sano", "rango": "2024-07-01/2024-07-15"},
    {"zona": "Castilla_Valladolid", "lat": 41.65, "lon": -4.72, "estado": "sano", "rango": "2024-05-01/2024-05-15"},
    {"zona": "Mato_Grosso_Brasil", "lat": -12.50, "lon": -55.80, "estado": "sano", "rango": "2024-02-01/2024-02-15"},
    {"zona": "Lleida_Frutales", "lat": 41.61, "lon": 0.62, "estado": "sano", "rango": "2024-06-01/2024-06-15"},
    {"zona": "Valle_Central_California", "lat": 36.75, "lon": -119.75, "estado": "sano", "rango": "2023-05-01/2023-05-15"},

    # --- ENFERMOS ---
    {"zona": "Sequia_Catalunya_2024", "lat": 41.80, "lon": 1.15, "estado": "enfermo_sequia", "rango": "2024-04-01/2024-04-15"},
    {"zona": "Sequia_Andalucia_2023", "lat": 37.55, "lon": -5.50, "estado": "enfermo_sequia", "rango": "2023-08-01/2023-08-15"},
    {"zona": "Estres_Termico_Texas", "lat": 33.50, "lon": -101.80, "estado": "enfermo_estres", "rango": "2023-07-01/2023-07-15"},

    # --- INCENDIOS ---
    {"zona": "Incendio_Texas_Smokehouse", "lat": 35.70, "lon": -101.40, "estado": "incendio", "rango": "2024-03-01/2024-03-15"},
    {"zona": "Incendio_Canada_Alberta", "lat": 54.10, "lon": -115.10, "estado": "incendio", "rango": "2023-06-01/2023-06-15"},
    {"zona": "Incendio_Valparaiso_Chile", "lat": -33.10, "lon": -71.50, "estado": "incendio", "rango": "2024-02-01/2024-02-15"}
]

miner = LandsatMiner()
bandas = ['red', 'green', 'blue', 'nir08', 'swir16', 'swir22']
datos_todos_los_campos = []

print("Iniciando procesamiento masivo...")

# ==========================================
# 3. BUCLE DE PROCESAMIENTO
# ==========================================
for parcela in anclas_estrategicas:
    print(f"Procesando {parcela['zona']}...")
    
    try:
        # Descargamos los datos
        ds = miner.fetch(lat=parcela['lat'], lon=parcela['lon'], radius=800, daterange=parcela['rango'])
        ds_single = ds[bandas].isel(time=0)
        
        # --- A. GUARDAR IMAGEN INTERACTIVA (.html) ---
        rgb_array = ds_single[['red', 'green', 'blue']].to_array(dim='band')
        rgb_norm = (rgb_array - 7000) / (35000 - 7000) # Normalización matemática
        rgb_norm = rgb_norm.clip(min=0, max=1)
        
        plot_interactivo = rgb_norm.hvplot.rgb(
            x='x', y='y', bands='band',
            title=f"Ref: {parcela['zona']} | {parcela['estado']}",
            width=500, height=500, geo=True
        )
        
        ruta_html = os.path.join(carpetas['html'], f"{parcela['zona']}.html")
        hvplot.save(plot_interactivo, ruta_html)
        
        # --- B. GUARDAR IMAGEN ORIGINAL (.tif) ---
        crs_value = ds_single.spatial_ref.item()
        ds_single.rio.write_crs(crs_value, inplace=True)
        
        ruta_tif = os.path.join(carpetas['tif'], f"datos_crudos_{parcela['zona']}.tif")
        ds_single.rio.to_raster(ruta_tif)
        
        # --- C. PROCESAR ESTADÍSTICAS PARA EL CSV ---
        df_pixeles = ds_single.to_dataframe().reset_index().dropna(subset=bandas)
        stats = df_pixeles[bandas].agg(['mean', 'std', 'min', 'max'])
        
        stats_fila = stats.unstack().to_frame().T
        stats_fila.columns = [f"{b}_{s}" for b, s in stats_fila.columns]
        
        stats_fila['id_campo'] = parcela['zona']
        stats_fila['estado'] = parcela['estado']
        stats_fila['fecha_captura'] = ds_single.time.values.astype('datetime64[D]')
        
        datos_todos_los_campos.append(stats_fila)
        print(f"  -> OK: {parcela['zona']} guardado correctamente.")

    except Exception as e:
        print(f"  -> ERROR en {parcela['zona']}: {e}")

# ==========================================
# 4. GUARDAR DATASET FINAL CSV
# ==========================================
if datos_todos_los_campos:
    df_final = pd.concat(datos_todos_los_campos, ignore_index=True)
    
    cols = ['id_campo', 'estado', 'fecha_captura'] + [c for c in df_final.columns if c not in ['id_campo', 'estado', 'fecha_captura']]
    df_final = df_final[cols]
    
    ruta_csv = os.path.join(carpetas['csv'], "dataset_machine_learning_completo.csv")
    df_final.to_csv(ruta_csv, index=False)
    
    print("\n¡Pipeline finalizado con éxito!")
    print(f"Dataset guardado en: {ruta_csv}")
    display(df_final.head())
else:
    print("No se pudo procesar ningún campo.")

Carpetas preparadas en: ./resultados_satelite/
Iniciando procesamiento masivo...
Procesando Cinturon_Maiz_Iowa...
  -> OK: Cinturon_Maiz_Iowa guardado correctamente.
Procesando Castilla_Valladolid...
  -> OK: Castilla_Valladolid guardado correctamente.
Procesando Mato_Grosso_Brasil...
  -> OK: Mato_Grosso_Brasil guardado correctamente.
Procesando Lleida_Frutales...
  -> OK: Lleida_Frutales guardado correctamente.
Procesando Valle_Central_California...
  -> OK: Valle_Central_California guardado correctamente.
Procesando Sequia_Catalunya_2024...
  -> OK: Sequia_Catalunya_2024 guardado correctamente.
Procesando Sequia_Andalucia_2023...
  -> OK: Sequia_Andalucia_2023 guardado correctamente.
Procesando Estres_Termico_Texas...
  -> OK: Estres_Termico_Texas guardado correctamente.
Procesando Incendio_Texas_Smokehouse...
  -> OK: Incendio_Texas_Smokehouse guardado correctamente.
Procesando Incendio_Canada_Alberta...
  -> OK: Incendio_Canada_Alberta guardado correctamente.
Procesando Incendio_V

,id_campo,estado,fecha_captura,red_mean,red_std,red_min,red_max,green_mean,green_std,green_min,...,nir08_min,nir08_max,swir16_mean,swir16_std,swir16_min,swir16_max,swir22_mean,swir22_std,swir22_min,swir22_max
0,Cinturon_Maiz_Iowa,sano,2024-07-04,10564.958984,2452.965820,7376.0,27026.0,10652.565430,2162.473633,7609.0,...,9955.0,30938.0,14247.468750,2964.228516,8665.0,27586.0,11780.354492,2568.345947,7895.0,21626.0
1,Castilla_Valladolid,sano,2024-05-03,24699.419922,2180.058838,15341.0,30726.0,24430.726562,2292.422607,14652.0,...,19351.0,32163.0,21791.691406,1187.338989,16919.0,25917.0,18139.019531,871.961731,14482.0,21079.0
2,Mato_Grosso_Brasil,sano,2024-02-04,43417.453125,725.289307,41332.0,45219.0,43580.410156,705.485962,41565.0,...,40463.0,44472.0,26191.560547,940.160583,23907.0,28899.0,20577.017578,789.021912,18631.0,22980.0
3,Lleida_Frutales,sano,2024-06-07,13787.592773,664.772278,11810.0,17438.0,13052.226562,516.053955,11633.0,...,15753.0,21750.0,15582.210938,703.222473,12814.0,18180.0,14100.297852,742.001160,11640.0,16374.0
4,Valle_Central_California,sano,2023-05-01,9976.636719,4329.360840,0.0,18803.0,9608.134766,4129.173828,0.0,...,0.0,18799.0,11582.378906,5077.862305,0.0,20440.0,10422.133789,4613.901367,0.0,23050.0
